In [0]:
data = [
    ("P001", "Arjun", 45, "Male", "Cardiology", "Kochi"),
    ("P002", "Meera", 32, "Female", "Neurology", "Kozhikode")
]
columns = ["patient_id", "name", "age", "gender", "department", "city"]
df = spark.createDataFrame(data, columns)
df.show()

+----------+-----+---+------+----------+---------+
|patient_id| name|age|gender|department|     city|
+----------+-----+---+------+----------+---------+
|      P001|Arjun| 45|  Male|Cardiology|    Kochi|
|      P002|Meera| 32|Female| Neurology|Kozhikode|
+----------+-----+---+------+----------+---------+



In [0]:
data = [
    ("P001", "Arjun", 45, "Male", "Cardiology", "Kochi"),
    ("P002", "Meera", 32, "Female", "Neurology", "Kozhikode")
]
columns = ["patient_id", "name", "age", "gender", "department", "city"]
df = spark.createDataFrame(data, columns)
df.display()

patient_id,name,age,gender,department,city
P001,Arjun,45,Male,Cardiology,Kochi
P002,Meera,32,Female,Neurology,Kozhikode


In [0]:
df_stud = spark.table("workspace.new.data_1")

df_stud.show()

df_stud.printSchema()

display(df_stud.limit(10))
display(df_stud.head(10))
print("There are ",len(df_stud.columns)," columns in the table.")

print("The data type of course is ",df_stud.schema["age"].dataType)

course_counts = df_stud.groupBy("course").count().orderBy("count", ascending=False)
print("Course appears the most is ",course_counts.first()[0])

+----------+-----+---+-----+------+
|student_id| name|age|marks|course|
+----------+-----+---+-----+------+
|      S001|Akhil| 21|   78|   BCA|
|      S002|Nisha| 22|   85|   BBA|
|      S003|Rohit| 20|   67|   BSc|
|      S004|Pooja| 23|   90|  BCom|
|      S005| Arun| 21|   72|   BCA|
+----------+-----+---+-----+------+

root
 |-- student_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: long (nullable = true)
 |-- marks: long (nullable = true)
 |-- course: string (nullable = true)



student_id,name,age,marks,course
S001,Akhil,21,78,BCA
S002,Nisha,22,85,BBA
S003,Rohit,20,67,BSc
S004,Pooja,23,90,BCom
S005,Arun,21,72,BCA


student_id,name,age,marks,course
S001,Akhil,21,78,BCA
S002,Nisha,22,85,BBA
S003,Rohit,20,67,BSc
S004,Pooja,23,90,BCom
S005,Arun,21,72,BCA


There are  5  columns in the table.
The data type of course is  LongType()
Course appears the most is  BCA


In [0]:
display(df_stud.count())

5

In [0]:
display(df_stud.columns)

_1
student_id
name
age
marks
course


In [0]:
df_stud.describe().show()


+-------+----------+-----+-----------------+-----------------+------+
|summary|student_id| name|              age|            marks|course|
+-------+----------+-----+-----------------+-----------------+------+
|  count|         5|    5|                5|                5|     5|
|   mean|      NULL| NULL|             21.4|             78.4|  NULL|
| stddev|      NULL| NULL|1.140175425099138|9.343446901438462|  NULL|
|    min|      S001|Akhil|               20|               67|   BBA|
|    max|      S005|Rohit|               23|               90|   BSc|
+-------+----------+-----+-----------------+-----------------+------+



In [0]:
display(df_stud.head(3))

student_id,name,age,marks,course
S001,Akhil,21,78,BCA
S002,Nisha,22,85,BBA
S003,Rohit,20,67,BSc


In [0]:
df_stud.agg({"marks": "sum"}).show()
df_stud.agg({"marks": "avg"}).show()
df_stud.agg({"marks": "min"}).show()
df_stud.agg({"marks": "mean"}).show()
df_stud.agg({"marks": "median"}).show()

+----------+
|sum(marks)|
+----------+
|       392|
+----------+

+----------+
|avg(marks)|
+----------+
|      78.4|
+----------+

+----------+
|min(marks)|
+----------+
|        67|
+----------+

+-----------+
|mean(marks)|
+-----------+
|       78.4|
+-----------+

+-------------+
|median(marks)|
+-------------+
|         78.0|
+-------------+



In [0]:
# %sql
# SELECT name, age FROM student WHERE age > 20;

df_stud.filter(df_stud.age > 20).display()
df_stud.filter(df_stud.age > 20).select("name", "age").display()

student_id,name,age,marks,course
S001,Akhil,21,78,BCA
S002,Nisha,22,85,BBA
S004,Pooja,23,90,BCom
S005,Arun,21,72,BCA


name,age
Akhil,21
Nisha,22
Pooja,23
Arun,21


In [0]:
df_stud.filter((df_stud.age > 20) & (df_stud.course == "BCA")).display()

student_id,name,age,marks,course
S001,Akhil,21,78,BCA
S005,Arun,21,72,BCA


In [0]:
df_stud.filter((df_stud.age > 20) & ((df_stud.course == "BCA") | (df_stud.course == "BBA"))).display()

student_id,name,age,marks,course
S001,Akhil,21,78,BCA
S002,Nisha,22,85,BBA
S005,Arun,21,72,BCA


In [0]:
from pyspark.sql import functions as F
# alias for rename
df_stud.select(
    F.count("name"),      # Result: 5 (skips the NULL)
    F.countDistinct("name"), # Result: 3 (Sci-Fi, Mystery, Romance)
    F.approx_count_distinct("age") # Fast estimate for huge data
).show()

+-----------+--------------------+--------------------------+
|count(name)|count(DISTINCT name)|approx_count_distinct(age)|
+-----------+--------------------+--------------------------+
|          5|                   5|                         4|
+-----------+--------------------+--------------------------+



In [0]:
df_stud.agg({"marks": "collect_list"}).show()
df_stud.agg({"marks": "collect_set"}).show()

+--------------------+
| collect_list(marks)|
+--------------------+
|[78, 85, 67, 90, 72]|
+--------------------+

+--------------------+
|  collect_set(marks)|
+--------------------+
|[78, 85, 67, 90, 72]|
+--------------------+



In [0]:
df_stud.agg({"name": "first"}).show()
df_stud.agg({"name": "last"}).show()

+-----------+
|first(name)|
+-----------+
|      Akhil|
+-----------+

+----------+
|last(name)|
+----------+
|      Arun|
+----------+

